In [1]:
# Example hugging face 
# packages installed with conda

In [2]:
!pip list


Package                   Version
------------------------- -----------
accelerate                1.14.0
aiofiles                  25.1.0
aiohappyeyeballs          2.6.2
aiohttp                   3.14.1
aiosignal                 1.4.0
annotated-doc             0.0.4
annotated-types           0.7.0
anyio                     4.14.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
backoff                   2.2.1
beautifulsoup4            4.15.0
bitsandbytes              0.49.2
bleach                    6.4.0
blis                      1.3.3
Brotli                    1.2.0
cachebox                  5.2.3
cached-property           1.5.2
catalogue                 2.0.10
certifi                   2024.2.2
cffi                      2.0.0
chardet                   7.4.3
charset-normalizer        3.4.7
click             

In [3]:
#!/usr/bin/env python
# coding: utf-8

# Advanced RAG on Hugging Face documentation using LangChain
# taken from https://huggingface.co/learn/cookbook/en/advanced_rag
# and also from https://python.langchain.com/v0.2/docs/integrations/document_loaders/url/

from langchain_community.document_loaders import UnstructuredURLLoader
#2025 from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

#from langchain.vectorstores import FAISS   #Facebook tool
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

#from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from typing import Optional, List, Tuple

#older 2025 from langchain.docstore.document import Document as LangchainDocument
from langchain_core.documents import Document as LangchainDocument

print('done importing')

/scratch/p4rodrig/job_51494953/ipykernel_1303750/1704544261.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader
/scratch/p4rodrig/job_51494953/miniforge3/envs/hf-t4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


done importing


In [4]:
def split_documents(
    chunk_size: int,
    knowledge_base: List[LangchainDocument],
    tokenizer_name: str, #EMBEDDING_MODEL_NAME
) -> List[LangchainDocument]:
    """
    Split documents into chunks of maximum size `chunk_size` tokens and return a list of documents.
    """
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained(tokenizer_name),
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size / 10),
        add_start_index=True,
        strip_whitespace=True,
        separators=MARKDOWN_SEPARATORS,
    )

    docs_processed = []
    for doc in knowledge_base:
        docs_processed += text_splitter.split_documents([doc])

    # Remove duplicates
    unique_texts = {}
    docs_processed_unique = []
    for doc in docs_processed:
        if doc.page_content not in unique_texts:
            unique_texts[doc.page_content] = True
            docs_processed_unique.append(doc)

    return docs_processed_unique
print('functions defined')

functions defined


In [5]:
# -----------------------------------------------
#   main starts here
# -----------------------------------------------

#if __name__ == '__main__':
if 1:
  #First set up loader and get web pages
  from langchain_community.document_loaders import UnstructuredURLLoader
  urls = [
    "https://slurm.schedmd.com/quickstart.html",
    "https://slurm.schedmd.com/man_index.html"
  ]

  loader = UnstructuredURLLoader(urls=urls)
  raw_pages = loader.load_and_split()

  #raw_pages is a list
  print('Num of raw pages after split:',len(raw_pages))

  #Second set up a model to split the web pages
  EMBEDDING_MODEL_NAME = "thenlper/gte-small"

  # We use a hierarchical list of separators specifically tailored for splitting Markdown documents
  # This list is taken from LangChain's MarkdownTextSplitter class
  MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
    ]

  docs_processed = split_documents(
    512,        # chunk size 
    raw_pages,  
    tokenizer_name=EMBEDDING_MODEL_NAME,
  )

  print('Length of docs:', len(docs_processed))

  #Third, set up embedding model and create vector database
  embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    multi_process=False,  #True,   #this might cause some fork issues?
    model_kwargs={"device": "cpu"},   # "cuda"},
    encode_kwargs={"normalize_embeddings": True},  # Set `True` for cosine similarity
  )

  KNOWLEDGE_VECTOR_DATABASE = FAISS.from_documents(
    docs_processed, embedding_model, distance_strategy=DistanceStrategy.COSINE
  )

#note viz pytorch 2.5.1, I get error 
    #AttributeError: module 'torch' has no attribute 'float8_e8m0fnu'

#so need pytroch 2.6 or lateest

Num of raw pages after split: 5
Length of docs: 12


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 230.80it/s]


In [6]:
! hf auth login --token hf_....   #Your Hug Face authorization token



Hint: A new version of huggingface_hub (1.21.0) is available! You are using version 1.20.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `c25` has been saved to /home/p4rodrig/.cache/huggingface/stored_tokens
Your token has been saved to /home/p4rodrig/.cache/huggingface/token
Login successful.
The current active token is: `c25`


In [7]:
if 1:
  #Now, embed a user query in the same space, show sample document
  user_query = "How to create a pipeline object?"
  query_vector = embedding_model.embed_query(user_query)

  print(f"\nStarting retrieval for {user_query=}...")
  retrieved_docs = KNOWLEDGE_VECTOR_DATABASE.similarity_search(query=user_query, k=5)
  #print(
  #  "\n==================================Top document=================================="
  #)
  #print(retrieved_docs[0].page_content)
  #print("==================================Metadata==================================")
  #print(retrieved_docs[0].metadata)


  #Now setup the hugging face pipeline
  import huggingface_hub
  from transformers import AutoTokenizer
  import transformers
  import torch

  #Set up model and tokenizer
  model="meta-llama/Meta-Llama-3-8B-Instruct"
  tokenizer = AutoTokenizer.from_pretrained(model)
  #print(tokenizer)


Starting retrieval for user_query='How to create a pipeline object?'...


In [8]:
if 1:
  #Set up prompt template with a place for context informatoin
  prompt_in_chat_format = [
    {
        "role": "system",
        "content": """Using the information contained in the context,
  give a comprehensive answer to the question.
  Respond only to the question asked, response should be concise and relevant to the question.
  Provide the number of the source document when relevant.
  If the answer cannot be deduced from the context, do not give an answer.""",
    },
    {
        "role": "user",
        "content": """Context:
  {context}
  ---
  Now here is the question you need to answer.

  Question: {question}""",
    },
  ]
  RAG_PROMPT_TEMPLATE = tokenizer.apply_chat_template(
    prompt_in_chat_format, tokenize=False, add_generation_prompt=True
  )
  #print(RAG_PROMPT_TEMPLATE)


  #set up actual prompt with context consisting of retreived docs
  retrieved_docs_text = [
    doc.page_content for doc in retrieved_docs
  ]  # We only need the text of the documents
  context = "\nExtracted documents:\n"
  context += "".join(
    [f"Document {str(i)}:::\n" + doc for i, doc in enumerate(retrieved_docs_text)]
  )

  final_prompt = RAG_PROMPT_TEMPLATE.format(
    question="How to create a slurm job?", context=context
  )

  print('----------------   Final Prompt -------------------------------------')
  print(final_prompt)

----------------   Final Prompt -------------------------------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Using the information contained in the context,
  give a comprehensive answer to the question.
  Respond only to the question asked, response should be concise and relevant to the question.
  Provide the number of the source document when relevant.
  If the answer cannot be deduced from the context, do not give an answer.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context:
  
Extracted documents:
Document 0:::
Figure 1. Slurm components

The entities managed by these Slurm daemons, shown in Figure 2, include nodes, the compute resource in Slurm, partitions, which group nodes into logical (possibly overlapping) sets, jobs, or allocations of resources assigned to a user for a specified amount of time, and job steps, which are sets of (possibly parallel) tasks within a job. The partitions can be considered job queues, each of which has an assortment of

In [9]:
if 1:
  #get device integer, for the pipeline definition below
  #torch.cuda.current_device() if torch.cuda.is_available() else 'cpu'
  device2use=0 if torch.cuda.is_available()  else -1  #-1 means default to cpu
  print('MYINFO device to use',device2use) 
    
    #, ' currdev', torch.cuda.current_device())   couldnt find gpu?

  #set up the function 
  my_pipe2 = transformers.pipeline(
    #"text-generation",
    model=model,
    #for gpu : torch_dtype=torch.float16,
    torch_dtype=torch.float32,  #for cpu use this
    device_map="auto",
    #device=device2use
  )
  print('pipeline2 defined')
  mem_allocated = torch.cuda.memory_allocated()
  print('MYINFO mem allocated before results:', mem_allocated)

  #now call the function with the prompt as input and other options
  results_list = my_pipe2(
    final_prompt,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=500, #num new tokens to generate
  )

  mem_allocated = torch.cuda.memory_allocated()
  print('MYINFO mem allocated aft results:', mem_allocated)

  for result in results_list:   #result is a python dict object
    print(' ----------------- Generated Text Result --------------------------')
    print(f"Result: {result['generated_text']}")

MYINFO device to use -1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [01:09<00:00,  4.17it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'do_sample', 'top_k', 'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


pipeline2 defined
MYINFO mem allocated before results: 0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


MYINFO mem allocated aft results: 0
 ----------------- Generated Text Result --------------------------
Result: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Using the information contained in the context,
  give a comprehensive answer to the question.
  Respond only to the question asked, response should be concise and relevant to the question.
  Provide the number of the source document when relevant.
  If the answer cannot be deduced from the context, do not give an answer.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context:
  
Extracted documents:
Document 0:::
Figure 1. Slurm components

The entities managed by these Slurm daemons, shown in Figure 2, include nodes, the compute resource in Slurm, partitions, which group nodes into logical (possibly overlapping) sets, jobs, or allocations of resources assigned to a user for a specified amount of time, and job steps, which are sets of (possibly parallel) tasks within a job. The partitions can be considered job 